# Sleep Data Exploration

Exploring Apple Watch sleep data to understand how motherhood affects sleep quality.

**Key dates:**
- Pregnancy 1: Oct 3, 2021 → May 7, 2022 (child born May 7, 2022)
- Pregnancy 2: Feb 17, 2025 → Nov 10, 2025 (child born Nov 10, 2025)
- Data starts: ~June 2021

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load data
with open('sleep_data.json') as f:
    sleep_raw = json.load(f)

df = pd.DataFrame(sleep_raw)
df['startDate'] = pd.to_datetime(df['startDate'].str[:-6])
df['endDate'] = pd.to_datetime(df['endDate'].str[:-6])
df['duration_min'] = (df['endDate'] - df['startDate']).dt.total_seconds() / 60

# Clean up sleep stage names
stage_map = {
    'HKCategoryValueSleepAnalysisAsleepCore': 'Core',
    'HKCategoryValueSleepAnalysisAsleepDeep': 'Deep',
    'HKCategoryValueSleepAnalysisAsleepREM': 'REM',
    'HKCategoryValueSleepAnalysisAsleepUnspecified': 'Asleep',
    'HKCategoryValueSleepAnalysisAwake': 'Awake',
    'HKCategoryValueSleepAnalysisInBed': 'InBed'
}
df['stage'] = df['value'].map(stage_map)

# Key dates
PREG1_START = pd.Timestamp('2021-10-03')
CHILD1_BORN = pd.Timestamp('2022-05-07')
PREG2_START = pd.Timestamp('2025-02-17')
CHILD2_BORN = pd.Timestamp('2025-11-10')

print(f'Records: {len(df)}')
print(f'Date range: {df.startDate.min()} to {df.startDate.max()}')
df.head()

In [ ]:
# Assign each sleep record to a "sleep night" (date of going to bed)
# If startDate is before 6 PM, it belongs to previous night
df['night_date'] = df['startDate'].apply(
    lambda x: x.date() if x.hour >= 18 else (x - timedelta(days=1)).date()
)
df['night_date'] = pd.to_datetime(df['night_date'])

In [ ]:
# Calculate nightly totals
# Total sleep = Core + Deep + REM + Asleep (not InBed, not Awake)
sleep_stages = ['Core', 'Deep', 'REM', 'Asleep']
asleep = df[df['stage'].isin(sleep_stages)]
awake = df[df['stage'] == 'Awake']

nightly = asleep.groupby('night_date').agg(
    total_sleep_min=('duration_min', 'sum'),
    sleep_segments=('duration_min', 'count'),
    first_sleep=('startDate', 'min'),
    last_wake=('endDate', 'max'),
).reset_index()

# Add awakening count per night
awake_counts = awake.groupby('night_date').agg(
    awakenings=('duration_min', 'count'),
    awake_min=('duration_min', 'sum')
).reset_index()

nightly = nightly.merge(awake_counts, on='night_date', how='left')
nightly['awakenings'] = nightly['awakenings'].fillna(0).astype(int)
nightly['awake_min'] = nightly['awake_min'].fillna(0)

# Sleep hours
nightly['sleep_hours'] = nightly['total_sleep_min'] / 60

# Time in bed
nightly['time_in_bed_min'] = (nightly['last_wake'] - nightly['first_sleep']).dt.total_seconds() / 60
nightly['efficiency'] = nightly['total_sleep_min'] / nightly['time_in_bed_min'] * 100

# Day of week
nightly['weekday'] = nightly['night_date'].dt.dayofweek  # 0=Mon
nightly['is_weekend'] = nightly['weekday'].isin([4, 5])  # Fri/Sat nights

# Bedtime hour
nightly['bedtime_hour'] = nightly['first_sleep'].dt.hour + nightly['first_sleep'].dt.minute / 60

# Season
month_to_season = {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',
                   6:'Summer',7:'Summer',8:'Summer',9:'Autumn',10:'Autumn',11:'Autumn'}
nightly['season'] = nightly['night_date'].dt.month.map(month_to_season)

# Life phase
def get_phase(date):
    if date < PREG1_START:
        return 'Pre-pregnancy'
    elif date < CHILD1_BORN:
        return 'Pregnancy 1'
    elif date < PREG2_START:
        return 'Postpartum 1'
    elif date < CHILD2_BORN:
        return 'Pregnancy 2'
    else:
        return 'Postpartum 2'

nightly['phase'] = nightly['night_date'].apply(get_phase)

print(f'Nights with sleep data: {len(nightly)}')
print(f'\nSleep hours stats:')
print(nightly['sleep_hours'].describe())
print(f'\nNights by phase:')
print(nightly['phase'].value_counts())

In [ ]:
# === MAIN PLOT: Sleep duration over time ===
fig, ax = plt.subplots(figsize=(18, 7))

# 7-day rolling average
nightly_sorted = nightly.sort_values('night_date')
nightly_sorted['sleep_7d'] = nightly_sorted['sleep_hours'].rolling(7, min_periods=3).mean()
nightly_sorted['sleep_30d'] = nightly_sorted['sleep_hours'].rolling(30, min_periods=10).mean()

# Plot daily as dots, rolling as line
ax.scatter(nightly_sorted['night_date'], nightly_sorted['sleep_hours'], 
           alpha=0.15, s=8, color='steelblue', label='Daily')
ax.plot(nightly_sorted['night_date'], nightly_sorted['sleep_30d'], 
        color='#E63946', linewidth=2.5, label='30-day average')

# Mark key events
events = [
    (PREG1_START, 'Pregnancy 1 starts', '#FF9F1C'),
    (CHILD1_BORN, 'Child 1 born', '#E63946'),
    (PREG2_START, 'Pregnancy 2 starts', '#FF9F1C'),
    (CHILD2_BORN, 'Child 2 born', '#E63946'),
]

for date, label, color in events:
    ax.axvline(x=date, color=color, linestyle='--', alpha=0.7, linewidth=1.5)
    ax.text(date, ax.get_ylim()[1] * 0.95, f'  {label}', 
            fontsize=9, color=color, rotation=0, va='top')

# Reference line at 7 hours (recommended minimum)
ax.axhline(y=7, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.text(nightly_sorted['night_date'].min(), 7.1, '7h recommended minimum', 
        fontsize=8, color='gray')

ax.set_ylabel('Sleep (hours)')
ax.set_title('Sleep Duration Over Time — The Motherhood Effect', fontsize=14, fontweight='bold')
ax.legend(loc='lower left')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# === Sleep by life phase ===
phase_order = ['Pre-pregnancy', 'Pregnancy 1', 'Postpartum 1', 'Pregnancy 2', 'Postpartum 2']
phase_stats = nightly.groupby('phase').agg(
    avg_sleep=('sleep_hours', 'mean'),
    std_sleep=('sleep_hours', 'std'),
    avg_awakenings=('awakenings', 'mean'),
    avg_efficiency=('efficiency', 'mean'),
    nights=('sleep_hours', 'count')
).reindex(phase_order)

print('Sleep by life phase:')
print(phase_stats.round(2))

# Box plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col, title in zip(axes, 
    ['sleep_hours', 'awakenings', 'efficiency'],
    ['Sleep Duration (hours)', 'Awakenings per Night', 'Sleep Efficiency (%)']
):
    data = [nightly[nightly['phase'] == p][col].dropna() for p in phase_order]
    bp = ax.boxplot(data, labels=[p.replace(' ', '\n') for p in phase_order], 
                    patch_artist=True)
    colors = ['#2A9D8F', '#FF9F1C', '#E63946', '#FF9F1C', '#E63946']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# === Awakenings over time ===
fig, ax = plt.subplots(figsize=(18, 5))

nightly_sorted['awakenings_30d'] = nightly_sorted['awakenings'].rolling(30, min_periods=10).mean()

ax.scatter(nightly_sorted['night_date'], nightly_sorted['awakenings'], 
           alpha=0.1, s=8, color='steelblue')
ax.plot(nightly_sorted['night_date'], nightly_sorted['awakenings_30d'], 
        color='#E63946', linewidth=2.5)

for date, label, color in events:
    ax.axvline(x=date, color=color, linestyle='--', alpha=0.7, linewidth=1.5)

ax.set_ylabel('Awakenings per night')
ax.set_title('Night Awakenings — Fragmented Sleep', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# === Weekday vs Weekend ===
fig, ax = plt.subplots(figsize=(10, 6))

for phase in phase_order:
    phase_data = nightly[nightly['phase'] == phase]
    weekday_avg = phase_data[~phase_data['is_weekend']]['sleep_hours'].mean()
    weekend_avg = phase_data[phase_data['is_weekend']]['sleep_hours'].mean()
    ax.scatter(['Weekday', 'Weekend'], [weekday_avg, weekend_avg], s=100, zorder=5)
    ax.plot(['Weekday', 'Weekend'], [weekday_avg, weekend_avg], label=phase)

ax.set_ylabel('Average Sleep (hours)')
ax.set_title('Weekday vs Weekend Sleep by Life Phase', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === Seasonal patterns ===
fig, ax = plt.subplots(figsize=(10, 6))

season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
for phase in phase_order:
    phase_data = nightly[nightly['phase'] == phase]
    seasonal = phase_data.groupby('season')['sleep_hours'].mean().reindex(season_order)
    ax.plot(season_order, seasonal.values, marker='o', label=phase)

ax.set_ylabel('Average Sleep (hours)')
ax.set_title('Seasonal Sleep Patterns by Life Phase', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === Bedtime drift ===
fig, ax = plt.subplots(figsize=(18, 5))

# Adjust bedtime for display (hours after midnight if < 18, else as-is)
nightly_sorted['bedtime_display'] = nightly_sorted['bedtime_hour'].apply(
    lambda h: h if h >= 18 else h + 24
)
nightly_sorted['bedtime_30d'] = nightly_sorted['bedtime_display'].rolling(30, min_periods=10).mean()

ax.scatter(nightly_sorted['night_date'], nightly_sorted['bedtime_display'], 
           alpha=0.1, s=8, color='steelblue')
ax.plot(nightly_sorted['night_date'], nightly_sorted['bedtime_30d'], 
        color='#E63946', linewidth=2.5)

for date, label, color in events:
    ax.axvline(x=date, color=color, linestyle='--', alpha=0.7, linewidth=1.5)

ax.set_ylabel('Bedtime (hour)')
ax.set_yticks([21, 22, 23, 24, 25, 26])
ax.set_yticklabels(['9 PM', '10 PM', '11 PM', '12 AM', '1 AM', '2 AM'])
ax.set_title('Bedtime Drift Over Time', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# === Resting Heart Rate (health impact indicator) ===
with open('resting_hr_data.json') as f:
    rhr_raw = json.load(f)

rhr = pd.DataFrame(rhr_raw)
rhr['date'] = pd.to_datetime(rhr['startDate'].str[:-6])
rhr['bpm'] = rhr['value'].astype(float)
rhr = rhr.sort_values('date')
rhr['bpm_30d'] = rhr['bpm'].rolling(30, min_periods=10).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

# Sleep
ax1.plot(nightly_sorted['night_date'], nightly_sorted['sleep_30d'], 
         color='#2A9D8F', linewidth=2.5, label='Sleep (30d avg)')
ax1.set_ylabel('Sleep (hours)')
ax1.set_title('Sleep Duration vs Resting Heart Rate', fontsize=14, fontweight='bold')
ax1.legend(loc='lower left')

# RHR
ax2.plot(rhr['date'], rhr['bpm_30d'], color='#E63946', linewidth=2.5, label='RHR (30d avg)')
ax2.set_ylabel('Resting Heart Rate (bpm)')
ax2.legend(loc='upper left')

for ax in [ax1, ax2]:
    for date, label, color in events:
        ax.axvline(x=date, color=color if 'born' in label else '#FF9F1C', 
                   linestyle='--', alpha=0.7, linewidth=1.5)

ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# === Summary statistics for the story hook ===
baseline_avg = nightly[nightly['phase'] == 'Pre-pregnancy']['sleep_hours'].mean()
total_deficit_hours = 0

for _, row in nightly_sorted.iterrows():
    if row['sleep_hours'] < baseline_avg:
        total_deficit_hours += (baseline_avg - row['sleep_hours'])

total_deficit_days = total_deficit_hours / 24

worst_week = nightly_sorted.set_index('night_date')['sleep_hours'].rolling('7D').mean().idxmin()
worst_week_avg = nightly_sorted.set_index('night_date')['sleep_hours'].rolling('7D').mean().min()

worst_night = nightly_sorted.loc[nightly_sorted['sleep_hours'].idxmin()]

print('=== STORY HOOK NUMBERS ===')
print(f'Baseline (pre-pregnancy) average: {baseline_avg:.1f} hours/night')
print(f'Total sleep deficit vs baseline: {total_deficit_hours:.0f} hours = {total_deficit_days:.0f} full days')
print(f'Worst week: around {worst_week.strftime("%B %d, %Y")} ({worst_week_avg:.1f} hrs avg)')
print(f'Worst single night: {worst_night["night_date"].strftime("%B %d, %Y")} ({worst_night["sleep_hours"]:.1f} hrs)')
print(f'\n=== BY PHASE ===')
for phase in phase_order:
    avg = nightly[nightly['phase'] == phase]['sleep_hours'].mean()
    awk = nightly[nightly['phase'] == phase]['awakenings'].mean()
    n = len(nightly[nightly['phase'] == phase])
    print(f'{phase}: {avg:.1f}h avg, {awk:.1f} awakenings/night ({n} nights)')

In [ ]:
# === Sleep stages breakdown (available from late 2022+) ===
stages_df = df[df['stage'].isin(['Core', 'Deep', 'REM'])]
if len(stages_df) > 0:
    stage_nightly = stages_df.groupby(['night_date', 'stage'])['duration_min'].sum().unstack(fill_value=0)
    stage_nightly = stage_nightly / 60  # to hours
    
    stage_monthly = stage_nightly.resample('M').mean()
    
    fig, ax = plt.subplots(figsize=(18, 6))
    stage_monthly.plot.bar(stacked=True, ax=ax, color=['#264653', '#2A9D8F', '#E9C46A'], width=0.8)
    ax.set_title('Sleep Stages Over Time (monthly average)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Hours')
    ax.set_xlabel('')
    ax.legend(title='Stage')
    
    # Simplify x labels
    labels = [item.get_text()[:7] for item in ax.get_xticklabels()]
    ax.set_xticklabels(labels, rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f'Stage data available from: {stages_df.startDate.min()}')
    print(f'Stage data available to: {stages_df.startDate.max()}')
else:
    print('No detailed sleep stage data found')

In [ ]:
# Save processed nightly data for React visualization
export_cols = ['night_date', 'sleep_hours', 'awakenings', 'awake_min', 
               'efficiency', 'bedtime_hour', 'weekday', 'is_weekend', 
               'season', 'phase', 'time_in_bed_min', 'sleep_segments']

nightly_export = nightly_sorted[export_cols].copy()
nightly_export['night_date'] = nightly_export['night_date'].dt.strftime('%Y-%m-%d')
nightly_export.to_json('nightly_sleep_processed.json', orient='records', indent=2)
print(f'Exported {len(nightly_export)} nights to nightly_sleep_processed.json')